# 21. Обучение multi-label R-BERT relation classifier

Модель обучается на gold-сущностях train и валидируется на gold-сущностях validation. Временные веса пишутся в `/content`, в Drive сохраняется только best checkpoint.

In [ ]:
from pathlib import Path
import os, runpy
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass
PROJECT_DIR = Path('/content/drive/MyDrive/NER_RuREBus_project')
if not PROJECT_DIR.exists(): PROJECT_DIR = Path.cwd()
os.environ['HF_HOME'] = '/content/huggingface_cache'
runpy.run_path(str(PROJECT_DIR / 'colab_bootstrap.py'))['bootstrap_project'](PROJECT_DIR)


In [ ]:
from datetime import datetime
from rurebus_ie.configuration import load_experiment_bundle
from rurebus_ie.training import persist_best_run, train_relation_experiment
CONFIG = PROJECT_DIR / 'configs/experiments/relation_classifier_global_v1.yaml'
bundle = load_experiment_bundle(CONFIG, project_root=PROJECT_DIR)
PERSISTENT_RUN = Path(bundle['experiment']['output_dir'])
LOCAL_RUN = Path('/content/rurebus_runs') / f"relations_global_v1_{datetime.now():%Y%m%d_%H%M%S}"
summary = train_relation_experiment(CONFIG, project_root=PROJECT_DIR, output_dir_override=LOCAL_RUN)
storage = persist_best_run(LOCAL_RUN, PERSISTENT_RUN)
print(f"Best epoch: {summary.best_epoch}; validation relation micro-F1: {summary.best_validation_f1:.6f}")
print(f"Сохранено в Drive: {storage['total_bytes'] / 2**30:.2f} GiB (только best).")


In [ ]:
import gc
try:
    import torch
    torch.cuda.empty_cache()
except Exception:
    pass
gc.collect()
